# ISA-95 Data Migration (Fabric PySpark)

This notebook runs source-to-entity and entity-to-entity (bridge) mappings using the exported Data Migration JSON configuration.

## 1) Configure Lakehouse paths

Update these paths to your Fabric environment.

In [ ]:
# Paths in Fabric Lakehouse Files
SCRIPT_PATH = '/lakehouse/default/Files/isa95/isa95_pyspark_migration.py'
CONFIG_PATH = '/lakehouse/default/Files/isa95/config/all_mappings_2026-03-03.json'
SOURCE_BASE_PATH = '/lakehouse/default/Files/isa95/source_tables'
OUTPUT_BASE_PATH = '/lakehouse/default/Files/isa95/output'
SOURCE_FORMAT = 'csv'  # csv | json | parquet

print('SCRIPT_PATH      =', SCRIPT_PATH)
print('CONFIG_PATH      =', CONFIG_PATH)
print('SOURCE_BASE_PATH =', SOURCE_BASE_PATH)
print('OUTPUT_BASE_PATH =', OUTPUT_BASE_PATH)
print('SOURCE_FORMAT    =', SOURCE_FORMAT)

## 2) Load migration module from script file

If the import fails, upload `isa95_pyspark_migration.py` to the path in `SCRIPT_PATH`.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('isa95_pyspark_migration', SCRIPT_PATH)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

run_migration = module.run_migration
print('Migration module loaded successfully')

## 3) Run migration

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

result = run_migration(
    spark=spark,
    config_path=CONFIG_PATH,
    source_base_path=SOURCE_BASE_PATH,
    output_base_path=OUTPUT_BASE_PATH,
    source_format=SOURCE_FORMAT,
)

print('Migration complete')
print('Successful:', result.successful)
print('Failed    :', result.failed)
print('Skipped   :', result.skipped)

## 4) Review skipped and failed mappings

In [ ]:
print('--- Failed mappings ---')
if result.failed_items:
    for item in result.failed_items:
        print('-', item)
else:
    print('None')

print('\n--- Skipped mappings ---')
if result.skipped_items:
    for item in result.skipped_items:
        print('-', item)
else:
    print('None')

## 5) Review generated outputs

In [ ]:
print('Output folders:')
for path in result.outputs:
    print('-', path)